# 03 — Evaluasi Akurasi (RQ1 & RQ2)

**Tujuan:** mengukur **akurasi** sistem RAG lokal dan membandingkannya dengan LLM tanpa RAG (*closed-book*).

| Bagian | Metrik | Menjawab |
|---|---|---|
| Retrieval | Recall@k, MRR, nDCG@k | RQ2 (strategi retrieval/embedding) |
| Jawaban | Exact Match, F1 (token) | RQ1 |
| Kualitas RAG | RAGAS: faithfulness, answer relevancy, context precision/recall | RQ1 |
| Baseline | EM & F1 **tanpa** retrieval | RQ1 (pembanding) |

> **Data uji:** `eval/gold_qa.csv`. Isi kolom `sumber` dengan **nama berkas** di `data/raw/ID_Regulations/` (mis. `dok_000123.pdf`) agar metrik retrieval bisa dihitung.
>
> **Parameter** (`CHUNK_SIZE`, `CHUNK_OVERLAP`, `EMBED_MODEL`) harus **sama** dengan notebook 02 saat indeks dibuat.

In [ ]:
import json
import math
import re
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# --- parameter (samakan dengan notebook 02) ---
CHUNK_SIZE = 512
CHUNK_OVERLAP = 64
EMBED_MODEL = "BAAI/bge-m3"
LLM_MODEL = "llama3.1:8b-instruct-q4_K_M"
K = 5

TAG = f"cs{CHUNK_SIZE}_ov{CHUNK_OVERLAP}_{EMBED_MODEL.split('/')[-1]}"
COLLECTION = re.sub(r"[^a-zA-Z0-9_]", "_", f"rag_{TAG}")

VECTOR_DIR = Path("vectorstore")
EVAL_DIR = Path("eval")
RESULT_DIR = EVAL_DIR / "results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("Tag         :", TAG)
print("Collection  :", COLLECTION)
print("LLM         :", LLM_MODEL)

## 0. Muat gold set

Baris template yang belum diisi (`<ganti...>`) otomatis dibuang.

In [ ]:
gold = pd.read_csv(EVAL_DIR / "gold_qa.csv")
gold = gold[~gold["sumber"].astype(str).str.contains("<ganti", na=False)].reset_index(drop=True)
print("Jumlah pertanyaan gold:", len(gold))
gold.head()

## 1. Muat indeks & model embedding

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(EMBED_MODEL)
client = chromadb.PersistentClient(path=str(VECTOR_DIR))
collection = client.get_collection(COLLECTION)
print("Collection:", COLLECTION, "| jumlah vektor:", collection.count())

## 2. Metrik retrieval (Recall@k, MRR, nDCG@k)

Sebuah hit dianggap benar bila **nama berkas rujukan** (`sumber`) muncul di antara top-k chunk yang diambil.

In [ ]:
def dcg(relevances):
    return sum(rel / math.log2(i + 2) for i, rel in enumerate(relevances))


def ndcg_at_k(retrieved_sources, relevant, k):
    rels = [1 if s in relevant else 0 for s in retrieved_sources[:k]]
    ideal = [1] * min(len(relevant), k)
    idcg = dcg(ideal)
    return dcg(rels) / idcg if idcg > 0 else 0.0


def retrieve(query, k):
    q_emb = model.encode([query], normalize_embeddings=True).tolist()
    res = collection.query(query_embeddings=q_emb, n_results=k)
    return res["metadatas"][0], res["documents"][0]


rows = []
for _, r in tqdm(gold.iterrows(), total=len(gold), desc="Retrieval"):
    metas, docs = retrieve(r["pertanyaan"], K)
    retrieved_sources = [m["source"] for m in metas]
    relevant = {str(r["sumber"])}

    hit = any(s in relevant for s in retrieved_sources)
    rr = 0.0
    for rank, s in enumerate(retrieved_sources, 1):
        if s in relevant:
            rr = 1 / rank
            break

    rows.append({
        "id": r["id"],
        "recall@k": int(hit),
        "mrr": rr,
        "ndcg@k": ndcg_at_k(retrieved_sources, relevant, K),
        "retrieved": "; ".join(retrieved_sources),
    })

retr = pd.DataFrame(rows)
print(f"Recall@{K} : {retr['recall@k'].mean():.3f}")
print(f"MRR       : {retr['mrr'].mean():.3f}")
print(f"nDCG@{K}   : {retr['ndcg@k'].mean():.3f}")

## 3. Generasi jawaban dengan RAG (Local LLM via Ollama)

Pipeline: pertanyaan → retrieval top-k → susun prompt berkonteks → LLM lokal menjawab **hanya** dari konteks.

In [ ]:
import ollama

PROMPT_TEMPLATE = (
    "Anda asisten kepatuhan. Jawab HANYA berdasarkan konteks di bawah.\n"
    "Jika jawaban tidak ada di konteks, katakan 'Tidak ditemukan dalam dokumen'.\n"
    "Sertakan rujukan nama dokumen/berkas.\n\n"
    "Konteks:\n{context}\n\n"
    "Pertanyaan: {question}\nJawaban:"
)


def answer(question, k=K):
    metas, docs = retrieve(question, k)
    context = "\n\n".join(f"[{m['source']} hal.{m['page']}] {d}" for m, d in zip(metas, docs))
    resp = ollama.chat(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": PROMPT_TEMPLATE.format(context=context, question=question)}],
    )
    return resp["message"]["content"], docs, metas

In [ ]:
gen_rows = []
for _, r in tqdm(gold.iterrows(), total=len(gold), desc="Generasi RAG"):
    ans, docs, metas = answer(r["pertanyaan"])
    gen_rows.append({
        "id": r["id"],
        "question": r["pertanyaan"],
        "answer": ans,
        "ground_truth": str(r["jawaban_referensi"]),
        "contexts": docs,
        "retrieved_sources": [m["source"] for m in metas],
    })

gen = pd.DataFrame(gen_rows)
gen.drop(columns=["contexts"]).head()

## 4. Exact Match & F1 (token)

Metrik sederhana berbasis tumpang tindih kata setelah normalisasi (huruf kecil, buang tanda baca). Cocok sebagai pembanding cepat, **bukan** pengganti penilaian semantik RAGAS.

In [ ]:
def normalize(s):
    s = str(s).lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    return " ".join(s.split())


def exact_match(pred, gold_text):
    return int(normalize(pred) == normalize(gold_text))


def f1_token(pred, gold_text):
    p_tokens = normalize(pred).split()
    g_tokens = normalize(gold_text).split()
    if not p_tokens or not g_tokens:
        return 0.0
    common = set(p_tokens) & set(g_tokens)
    if not common:
        return 0.0
    overlap = sum(min(p_tokens.count(t), g_tokens.count(t)) for t in common)
    precision = overlap / len(p_tokens)
    recall = overlap / len(g_tokens)
    return 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0


em = gen.apply(lambda r: exact_match(r["answer"], r["ground_truth"]), axis=1).mean()
f1 = gen.apply(lambda r: f1_token(r["answer"], r["ground_truth"]), axis=1).mean()
print(f"Dengan RAG  ->  EM: {em:.3f}   F1: {f1:.3f}")

## 5. Baseline: LLM tanpa RAG (*closed-book*) — untuk RQ1

In [ ]:
base_rows = []
for _, r in tqdm(gold.iterrows(), total=len(gold), desc="Baseline tanpa RAG"):
    resp = ollama.chat(model=LLM_MODEL, messages=[
        {"role": "user", "content": f"Jawab singkat: {r['pertanyaan']}"}
    ])
    base_rows.append({
        "id": r["id"],
        "answer": resp["message"]["content"],
        "ground_truth": str(r["jawaban_referensi"]),
    })

base = pd.DataFrame(base_rows)
base_em = base.apply(lambda r: exact_match(r["answer"], r["ground_truth"]), axis=1).mean()
base_f1 = base.apply(lambda r: f1_token(r["answer"], r["ground_truth"]), axis=1).mean()
print(f"Baseline    ->  EM: {base_em:.3f}   F1: {base_f1:.3f}")
print(f"Dengan RAG  ->  EM: {em:.3f}   F1: {f1:.3f}")

## 6. RAGAS (faithfulness, answer relevancy, context precision/recall)

Menggunakan LLM & embedding **lokal** (Ollama). Bila belum terpasang, blok ini akan menampilkan instruksi, bukan berhenti.

Prasyarat: `pip install ragas langchain-ollama` dan server Ollama berjalan.

In [ ]:
ragas_scores = {}
try:
    from datasets import Dataset
    from ragas import evaluate
    from ragas.metrics import (
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    )
    from langchain_ollama import ChatOllama, OllamaEmbeddings

    ds = Dataset.from_dict({
        "question": gen["question"].tolist(),
        "answer": gen["answer"].tolist(),
        "contexts": gen["contexts"].tolist(),
        "ground_truth": gen["ground_truth"].tolist(),
    })

    result = evaluate(
        ds,
        metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
        llm=ChatOllama(model=LLM_MODEL, temperature=0),
        embeddings=OllamaEmbeddings(model="nomic-embed-text"),
    )
    ragas_scores = result.to_pandas().mean(numeric_only=True).to_dict()
    print(ragas_scores)
except Exception as e:
    print("RAGAS belum siap dijalankan:", type(e).__name__, "-", e)
    print("Pastikan: pip install ragas langchain-ollama; Ollama berjalan; model tersedia.")

## 7. Simpan hasil (ketertelusuran)

Ringkasan disimpan ke `eval/results/` supaya setiap angka di BAB IV bisa dilacak.

In [ ]:
summary = {
    "date": date.today().isoformat(),
    "tag": TAG,
    "llm": LLM_MODEL,
    "embed_model": EMBED_MODEL,
    "k": K,
    "n_questions": int(len(gold)),
    "retrieval": {
        "recall@k": float(retr["recall@k"].mean()),
        "mrr": float(retr["mrr"].mean()),
        "ndcg@k": float(retr["ndcg@k"].mean()),
    },
    "answer": {
        "rag_exact_match": float(em),
        "rag_f1": float(f1),
        "baseline_exact_match": float(base_em),
        "baseline_f1": float(base_f1),
    },
    "ragas": {k: float(v) for k, v in ragas_scores.items()} if ragas_scores else None,
}

(RESULT_DIR / f"accuracy_{TAG}.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"
)
retr.to_csv(RESULT_DIR / f"retrieval_{TAG}.csv", index=False)
gen.drop(columns=["contexts"]).to_csv(RESULT_DIR / f"generation_{TAG}.csv", index=False)
base.to_csv(RESULT_DIR / f"baseline_{TAG}.csv", index=False)

print(json.dumps(summary, indent=2, ensure_ascii=False))

## Langkah berikutnya

1. Ulangi notebook 02 + 03 dengan `CHUNK_SIZE`/`EMBED_MODEL` lain → tabel perbandingan RQ2.
2. Jalankan `notebook/04_keamanan.ipynb` → metrik keamanan RQ3.
3. Catat latensi & puncak VRAM tiap run → RQ4.
4. Tempel angka ringkasan ke tabel hasil di BAB IV.

> **Penting:** `CHUNK_SIZE`, `EMBED_MODEL`, dan `K` menentukan nama `TAG`. Pastikan notebook 02 dan 03 memakai nilai yang sama agar collection yang di-query benar.